In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel('data.xlsx')
country_code = pd.read_excel('Country-Code.xlsx')
df.shape

(9551, 19)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9550 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Price range           9551 non-null   int64  
 15  Aggregate rating     

In [4]:
df.isnull().sum()

Restaurant ID           0
Restaurant Name         1
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df['Restaurant ID'].duplicated().sum()

np.int64(0)

In [7]:
## Identify duplicates and remove them
df = df.drop_duplicates()
df = df.dropna(subset=['Restaurant Name'])
df['Cuisines'] = df['Cuisines'].fillna('unknown')
df = df.merge(country_code, on='Country Code', how='left')
df.shape

(9550, 20)

In [8]:
##Performing EDA (first set)
#Step 3: Geographical distribution — max/min restaurant cities

city_counts = df['City'].value_counts()
city_counts

City
New Delhi    5473
Gurgaon      1118
Noida        1080
Faridabad     251
Ghaziabad      25
             ... 
Lincoln         1
Monroe          1
Potrero         1
Weirton         1
Miller          1
Name: count, Length: 141, dtype: int64

In [9]:
city_counts.idxmax(), city_counts.max()

('New Delhi', 5473)

In [10]:
#Step 4: Franchise with most national presence
franchise_reach = df.groupby('Restaurant Name')['City'].nunique().sort_values(ascending=False)
franchise_reach.head(10)

Restaurant Name
Barbeque Nation      22
Pizza Hut            12
KFC                   8
McDonald's            7
The Yellow Chilli     7
Pind Balluchi         7
Domino's Pizza        7
Mocha                 7
TGI Friday's          6
Farzi Cafe            6
Name: City, dtype: int64

In [11]:
#Step 5: Table booking ratio
df['Has Table booking'].value_counts()

Has Table booking
No     8392
Yes    1158
Name: count, dtype: int64

In [12]:
#Step 6: Percentage of restaurants providing online delivery
df['Has Online delivery'].value_counts(normalize=True) * 100

Has Online delivery
No     74.335079
Yes    25.664921
Name: proportion, dtype: float64

In [13]:
#Step 7: Difference in votes — delivery vs non-delivery
df.groupby('Has Online delivery')['Votes'].sum()

Has Online delivery
No     979962
Yes    517914
Name: Votes, dtype: int64

In [14]:
#Performing EDA (second set)

#Step 1: Top 10 cuisines served across cities
Cuisines = df.copy()
Cuisines = Cuisines.assign(Cuisines=Cuisines['Cuisines'].str.split(","))
Cuisines = Cuisines.explode('Cuisines')
Cuisines['Cuisines'] = Cuisines['Cuisines'].str.strip()
top10 = Cuisines['Cuisines'].value_counts().head(10)
top10

Cuisines
North Indian    3959
Chinese         2735
Fast Food       1986
Mughlai          995
Italian          763
Bakery           745
Continental      735
Cafe             703
Desserts         653
South Indian     636
Name: count, dtype: int64

In [15]:
#Step 2: Max/min cuisines per restaurant + most served cuisine per city
cuisine_count = df['Cuisines'].apply(lambda x: len(x.split(',')))
cuisine_count.max(), cuisine_count.min()

(8, 1)

In [16]:
top_cuisine_per_city = Cuisines.groupby('City')['Cuisines'].agg(lambda x: x.value_counts().idxmax())
top_cuisine_per_city

City
Abu Dhabi                   Indian
Agra                  North Indian
Ahmedabad              Continental
Albany                       Steak
Allahabad             North Indian
                        ...       
Weirton                     Burger
Wellington City               Cafe
Winchester Bay              Burger
Yorkton                      Asian
€¡stanbul          Restaurant Cafe
Name: Cuisines, Length: 141, dtype: object

In [17]:
#Step 3: Distribution of cost across restaurants
df['Average Cost for two'].describe()

count      9550.000000
mean       1199.252565
std       16122.026663
min           0.000000
25%         250.000000
50%         400.000000
75%         700.000000
max      800000.000000
Name: Average Cost for two, dtype: float64

In [18]:
df.groupby('Currency')['Average Cost for two'].agg(['mean', 'max', 'count'])

,mean,max,count
Currency,,,
Botswana Pula(P),1606.818182,6000,22
Brazilian Real(R$),134.666667,400,60
Dollar($),31.510373,500,482
Emirati Diram(AED),166.416667,500,60
Indian Rupees(Rs.),623.349902,8000,8651
Indonesian Rupiah(IDR),281190.476190,800000,21
NewZealand($),69.750000,200,40
Pounds(å£),47.812500,230,80
Qatari Rial(QR),223.750000,550,20


In [19]:
#Step 4: How ratings are distributed among various factors
df.groupby('Price range')['Aggregate rating'].mean()

Price range
1    1.999887
2    2.941054
3    3.683085
4    3.817918
Name: Aggregate rating, dtype: float64

In [20]:
df.groupby('Has Table booking')['Aggregate rating'].mean()

Has Table booking
No     2.559175
Yes    3.441969
Name: Aggregate rating, dtype: float64

In [21]:
df.groupby('Has Online delivery')['Aggregate rating'].mean()

Has Online delivery
No     2.465066
Yes    3.248837
Name: Aggregate rating, dtype: float64

In [23]:
#Step 5: Factors that may affect ratings
df['NumCuisines'] = cuisine_count
df[['Aggregate rating', 'Average Cost for two', 'Votes', 'Price range', 'NumCuisines']].corr()['Aggregate rating']

Aggregate rating        1.000000
Average Cost for two    0.051797
Votes                   0.313598
Price range             0.437874
NumCuisines             0.249271
Name: Aggregate rating, dtype: float64

In [25]:
#Step 6/7: Export cleaned data for Tableau
df.to_csv('cleaned_data.csv', index=False)